### Use of nn module in PyTorch to create neural networks

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# define a neural network class and also write why is it use
class MyNeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MyNeuralNetwork, self).__init__()
        # define layers
        self.fc1 = nn.Linear(input_size, hidden_size)  # first fully connected layer
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # second fully connected layer
        self.fc3 = nn.Linear(hidden_size, output_size)  # third fully connected layer
        # dropout layer to prevent overfitting
        self.dropout = nn.Dropout(p=0.5)  # dropout with probability of 0.5

    # forward pass of the neural network
    # The forward method defines how the input data flows through the network layers. It takes an input tensor x, applies the first fully connected layer (fc1), followed by a ReLU activation function, then applies dropout to prevent overfitting. The process is repeated for the second fully connected layer (fc2) and finally, the output is produced by the third fully connected layer (fc3). The output is returned as the final result of the forward pass.
    def forward(self, x):
        x = F.relu(self.fc1(x))  # apply first layer and ReLU activation
        x = self.dropout(x)  # apply dropout
        x = F.relu(self.fc2(x))  # apply second layer and ReLU activation
        x = self.dropout(x)  # apply dropout
        x = self.fc3(x)  # apply third layer (output layer)
        return x  # return the output
        

In [4]:
# create instance of the neural network
model = MyNeuralNetwork(input_size=784, hidden_size=128, output_size=10)  # example for MNIST dataset
print(model)  # print the model architecture

MyNeuralNetwork(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
# Create datasets and dataloaders for training and testing
# create model
model2 = MyNeuralNetwork(input_size=5, hidden_size=10, output_size=2)
print(model2)

MyNeuralNetwork(
  (fc1): Linear(in_features=5, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=10, bias=True)
  (fc3): Linear(in_features=10, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [7]:
# Create the model
model = MyNeuralNetwork(input_size=784, hidden_size=256, output_size=10)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(model)

MyNeuralNetwork(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [8]:
import torch.optim as optim

# Loss function (depends on your task)
criterion = nn.CrossEntropyLoss()  # For classification
# criterion = nn.MSELoss()  # For regression

# Optimizer (updates weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [9]:
from torch.utils.data import DataLoader, TensorDataset

# Example: Create dummy data (replace with your actual data)
# Assuming you have features (X) and labels (y)
X = torch.randn(1000, 784)  # 1000 samples, 784 features
y = torch.randint(0, 10, (1000,))  # 1000 labels (0-9)

# Create dataset and dataloader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [18]:
X.shape
y.shape


torch.Size([1000])

In [10]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    
    for batch_idx, (data, targets) in enumerate(dataloader):
        # Move data to device
        data, targets = data.to(device), targets.to(device)
        
        # Forward pass
        outputs = model(data)
        loss = criterion(outputs, targets)
        
        # Backward pass
        optimizer.zero_grad()  # Clear previous gradients
        loss.backward()        # Compute gradients
        optimizer.step()       # Update weights
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [1/10], Loss: 2.3188
Epoch [2/10], Loss: 2.1907
Epoch [3/10], Loss: 1.9553
Epoch [4/10], Loss: 1.4772
Epoch [5/10], Loss: 0.8826
Epoch [6/10], Loss: 0.4759
Epoch [7/10], Loss: 0.3116
Epoch [8/10], Loss: 0.2501
Epoch [9/10], Loss: 0.1755
Epoch [10/10], Loss: 0.1344


In [11]:
# Evaluation mode (disables dropout)
model.eval()

with torch.no_grad():  # Disable gradient computation
    correct = 0
    total = 0
    
    for data, targets in dataloader:
        data, targets = data.to(device), targets.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 100.00%
